# Lección 8.1 — Encontrar estructura sin etiquetas

> **Módulo 8 · Aprendizaje no supervisado** · Nivel 3 — Practicante
> ⏱️ Tiempo estimado: 80–100 min (lección con ejercicios intercalados ≈ 55 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 1.3 (tipos de aprendizaje), Lección 2.3 (escalado), Lección 2.8 (maldición de la dimensionalidad)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar qué significa aprender **sin etiquetas** y en qué se diferencia del aprendizaje supervisado.
- Reconocer las cuatro grandes tareas no supervisadas: **agrupar**, **reducir dimensiones**, **detectar anomalías** y **recomendar**.
- Entender por qué todo depende de una idea: **qué tan parecidos son dos datos** (la distancia), y por qué escalar es obligatorio.
- Desconfiar sanamente: un algoritmo de agrupamiento **siempre** encuentra grupos, existan o no.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: ordenar un cajón sin instrucciones

Imagina que heredas una caja con **500 botones** sueltos. Nadie te dice qué categorías existen. Aun así, en pocos minutos los tienes separados en montoncitos: los grandes de abrigo, los pequeños de camisa, los dorados, los de madera…

Nadie te dio **etiquetas**. Tú **descubriste** la estructura mirando qué botones se **parecen** entre sí.

Eso es el **aprendizaje no supervisado**:

| | Supervisado (Módulos 4–7) | No supervisado (este módulo) |
|---|---|---|
| Qué recibe el modelo | Ejemplos **con** la respuesta (`X` y `y`) | Solo ejemplos (`X`), **sin** respuesta |
| Qué aprende | Una regla para predecir `y` | La **forma** de los datos: grupos, ejes, rarezas |
| Analogía | Un alumno con las soluciones del libro | Un explorador sin mapa |
| Cómo se evalúa | Comparando con la respuesta real | **Difícil**: no hay respuesta real con qué comparar |

### ✋ Ejercicio 1 — ¿Supervisado o no supervisado?
Para cada caso, decide si es **supervisado (S)** o **no supervisado (N)**:

1. Predecir el precio de una casa a partir de casas ya vendidas.
2. Dividir a los clientes de un supermercado en grupos según lo que compran, sin saber de antemano qué grupos existen.
3. Detectar transacciones bancarias "raras" cuando nadie ha marcado cuáles son fraude.
4. Clasificar correos como spam o no spam usando miles de correos que los usuarios ya marcaron.
5. Resumir 64 medidas de cada paciente en 2 números para poder dibujarlos en una gráfica.

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuesta</b></summary>

1. **S** — hay respuesta conocida (el precio de venta).
2. **N** — es **agrupamiento** (*clustering*): no sabemos qué grupos existen.
3. **N** — es **detección de anomalías**: nadie etiquetó qué es fraude.
4. **S** — los usuarios ya etiquetaron los correos.
5. **N** — es **reducción de dimensionalidad**: comprimir sin una respuesta que predecir.

Pista para el futuro: si existe una columna "la respuesta correcta", casi siempre es supervisado.
</details>

---
## 2. 📖 El concepto: las cuatro grandes tareas

El aprendizaje no supervisado no es un único modelo, es una **familia de tareas**. Este módulo las recorre todas:

| Tarea | Pregunta que responde | Ejemplo de negocio | Lecciones |
|---|---|---|---|
| **Agrupamiento** (*clustering*) | ¿Qué grupos naturales hay? | Segmentar clientes | 8.2, 8.3 |
| **Reducción de dimensionalidad** | ¿Cómo resumo muchas columnas en pocas? | Dibujar datos de 64 columnas en 2D | 8.4, 8.5 |
| **Detección de anomalías** | ¿Qué datos no se parecen a nada? | Fraude, fallas de máquinas | 8.6 |
| **Recomendación** | ¿Qué le gustará a esta persona? | "Quienes vieron esto también vieron…" | 8.7 |

¿Por qué es tan útil si "no predice nada"? Porque **las etiquetas son caras**. Etiquetar 1 millón de transacciones como fraude/no fraude cuesta meses de trabajo humano; tenerlas sin etiquetar es gratis. El no supervisado saca valor de esos datos "en bruto".

### ✋ Ejercicio 2 — Empareja la tarea
Una tienda en línea tiene estos problemas. ¿Qué tarea de la tabla usarías para cada uno?

- a) "Queremos mandar correos distintos a distintos tipos de compradores, pero no sabemos qué tipos hay."
- b) "Un pedido de 400 televisores a una dirección nueva a las 3 a. m. nos parece sospechoso."
- c) "En la página de cada producto queremos mostrar 5 productos que podrían interesarte."
- d) "Tenemos 300 métricas por cliente y el equipo de marketing no entiende nada en una tabla así."

✍️ *Mis respuestas:* a → _ · b → _ · c → _ · d → _

<details><summary><b>👀 Ver respuesta</b></summary>

- a) **Agrupamiento** (segmentación de clientes).
- b) **Detección de anomalías**.
- c) **Recomendación**.
- d) **Reducción de dimensionalidad** (resumir 300 métricas en unas pocas que se puedan graficar).

En la vida real se combinan: por ejemplo, reducir dimensiones **antes** de agrupar, o agrupar para recomendar.
</details>

---
## 3. 📖 Cómo funciona: todo depende de la palabra "parecido"

Para agrupar botones tú usaste un criterio de **parecido**: tamaño, color, material. Los algoritmos necesitan lo mismo, pero en números: la **distancia** entre dos datos. Cuanto **menor** la distancia, más parecidos son.

La más usada es la **distancia euclidiana**: la longitud de la línea recta entre dos puntos (la de la regla, la de siempre). Ya la usaste con KNN en la Lección 5.3.

Veamos tres clientes de un gimnasio, descritos por su **edad** (años) y su **ingreso mensual** (pesos):

In [ ]:
clientes = pd.DataFrame({
    "edad":    [25, 62, 27],
    "ingreso": [30_000, 31_000, 45_000],
}, index=["Ana", "Beto", "Carla"])
clientes

### 🔮 Ejercicio 3 — ¿Quién se parece más a Ana?
Mirando la tabla con tu sentido común: ¿Ana se parece más a **Beto** (casi el mismo ingreso, pero 37 años mayor) o a **Carla** (casi la misma edad, pero gana 15,000 más)?

Ahora piensa como la computadora: calcula la distancia "en línea recta" usando los números tal cual. ¿Quién sale más cerca?

✍️ *Mi apuesta:*

In [ ]:
def distancia(a, b):
    return np.sqrt(((a - b) ** 2).sum())   # distancia euclidiana

ana = clientes.loc["Ana"]
for otro in ["Beto", "Carla"]:
    print(f"Distancia Ana–{otro}: {distancia(ana, clientes.loc[otro]):>10,.1f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Para la computadora, Ana está **mucho más cerca de Beto** (≈1,000) que de Carla (≈15,000). ¿Por qué? Porque el ingreso se mide en **miles** y la edad en **decenas**: una diferencia de 1,000 pesos "pesa" más que una diferencia de 37 años. La edad prácticamente **no cuenta**.

¿Es Beto más parecido a Ana? Depende del negocio, pero lo seguro es que la computadora **no lo decidió por una buena razón**: lo decidió por las **unidades**.
</details>

### 📖 La solución: escalar (Lección 2.3)
`StandardScaler` pone cada columna en la misma escala (promedio 0, desviación 1), así cada característica tiene **la misma voz** en la distancia.

El escalador se **aprende** con todos los clientes del gimnasio (no solo con estos 3): así sabe cuánto varían de verdad la edad y el ingreso.

In [ ]:
# 500 socios del gimnasio: la edad varía unos ±12 años y el ingreso unos ±12,000 pesos
rng_gym = np.random.default_rng(1)
socios = pd.DataFrame({"edad": rng_gym.normal(40, 12, 500), "ingreso": rng_gym.normal(35_000, 12_000, 500)})

escalador = StandardScaler().fit(socios)          # aprende promedio y desviación de cada columna
escalados = pd.DataFrame(escalador.transform(clientes),
                         index=clientes.index, columns=clientes.columns)
print(escalados.round(2), "\n")

ana_e = escalados.loc["Ana"]
for otro in ["Beto", "Carla"]:
    print(f"Distancia Ana–{otro} (escalado): {distancia(ana_e, escalados.loc[otro]):.2f}")

<details><summary><b>👀 ¿Qué cambió?</b></summary>

Después de escalar, Ana queda **mucho más cerca de Carla** (≈1.2) que de Beto (≈3.4). En el gimnasio la edad varía unos ±11 años y el ingreso unos ±12,600 pesos: medidos en esas "desviaciones típicas", 37 años de diferencia son más de 3 desviaciones, mientras que 15,000 pesos son apenas 1.2. Ahora la edad sí cuenta.

La lección: **en no supervisado no hay una respuesta correcta que corrija tus decisiones**. Si no escalas, el algoritmo agrupará por "la columna con números más grandes" y nadie te avisará. Escalar no es un detalle técnico: **decide qué significa "parecido"**.
</details>

### ✋ Ejercicio 4 — Elige qué cuenta
Un analista quiere agrupar clientes y tiene estas columnas: `edad`, `ingreso`, `compras_al_mes`, `id_cliente`, `codigo_postal`.

1. ¿Qué columna **nunca** debería entrar en la distancia?
2. `codigo_postal` es un número (p. ej. 06700). ¿Tiene sentido restar dos códigos postales?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **`id_cliente`**: es un identificador, no describe al cliente. Si entra, dos clientes con ID 1001 y 1002 parecerán "gemelos".
2. **No.** El código 06700 no está "más cerca" de 06701 que de 44100 en ningún sentido útil (podrían ser barrios muy distintos). Es una variable **categórica disfrazada de número** (Lección 2.1). Habría que convertirla en algo con significado (región, distancia a la tienda…) o dejarla fuera.

En no supervisado, **elegir las columnas** es elegir qué significa "parecido". Es la decisión más importante del análisis.
</details>

---
## 4. 📖 ¿Qué pasa si… no hay grupos?

Vamos a ver la primera tarea en acción: el agrupamiento. Usaremos **K-means** (lo estudiaremos a fondo en la Lección 8.2). Por ahora basta saber que le dices **cuántos grupos quieres** (`k`) y él reparte los datos.

Primero, datos **con** grupos claros:

In [ ]:
X_grupos, _ = make_blobs(n_samples=300, centers=3, cluster_std=0.8, random_state=4)

grupos = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(X_grupos)
plt.scatter(*X_grupos.T, c=grupos, cmap="viridis", s=20, edgecolor="k")
plt.title("Datos con 3 grupos reales → K-means con k=3"); plt.show()

Perfecto: encontró los tres grupos que se ven a simple vista.

### 🔮 Ejercicio 5 — La trampa
Ahora generamos puntos **completamente al azar**, repartidos de manera uniforme en un cuadrado: **no hay ningún grupo**. Le pedimos a K-means `k=3`.

¿Qué crees que hará?
- a) Dará un error: "no hay grupos".
- b) Pondrá todos los puntos en un solo grupo.
- c) Partirá el cuadrado en 3 "grupos" con total confianza.

✍️ *Mi apuesta:*

In [ ]:
rng = np.random.default_rng(0)
X_azar = rng.uniform(0, 10, size=(300, 2))

grupos_azar = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(X_azar)
plt.scatter(*X_azar.T, c=grupos_azar, cmap="viridis", s=20, edgecolor="k")
plt.title("Datos SIN grupos → K-means con k=3 igual 'encuentra' 3"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

**c)** K-means corta el cuadrado en 3 rebanadas y te devuelve 3 grupos como si nada. **Un algoritmo de agrupamiento siempre devuelve grupos**, aunque no existan.

Esta es la advertencia más importante del módulo. Si un analista te dice "encontramos 5 tipos de clientes", la pregunta correcta es: **"¿cómo sabes que esos grupos son reales y útiles, y no solo un corte arbitrario?"**
</details>

### 📖 Entonces, ¿cómo evaluamos sin etiquetas?
No hay una respuesta perfecta, pero hay tres herramientas:

| Herramienta | Qué mide | Ejemplo |
|---|---|---|
| **Métricas internas** | ¿Los grupos son compactos y están bien separados? | **Silueta** (Lección 8.2): de −1 a 1, más alto = grupos más claros |
| **Validación externa** | Si por casualidad tienes alguna etiqueta, ¿los grupos coinciden con ella? | *Adjusted Rand Index* (ARI): 1 = coinciden perfecto, ≈0 = como al azar |
| **Utilidad de negocio** | ¿Alguien puede **actuar** distinto con cada grupo? | "El grupo 2 responde a descuentos; el 3 no" |

La silueta se calcula solo con los datos y los grupos. Comparemos los dos casos anteriores:

### 🔮 Ejercicio 6
¿Qué datos tendrán la silueta **más alta**: los que tienen grupos reales o los aleatorios?

✍️ *Mi apuesta:*

In [ ]:
print(f"Silueta con grupos reales : {silhouette_score(X_grupos, grupos):.2f}")
print(f"Silueta con datos al azar : {silhouette_score(X_azar, grupos_azar):.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Grupos reales: ≈**0.69**. Datos al azar: ≈**0.38**. La silueta detecta que en el segundo caso los "grupos" están pegados unos a otros.

Pero ojo: 0.38 no es cero. Una silueta mediocre **no prueba** que no haya grupos, ni una alta prueba que sean útiles. Las métricas internas son una pista, no un veredicto. Por eso siempre las combinamos con el criterio de negocio.
</details>

---
## 5. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Los grupos se separan solo por una columna (p. ej. ingreso) | Columnas **sin escalar**: la de números grandes domina | `StandardScaler` antes de agrupar (Lección 2.3) |
| "Encontramos 7 segmentos" pero nadie sabe qué hacer con ellos | Grupos sin **utilidad de negocio** | Perfilar cada grupo; si no cambia ninguna decisión, sobran grupos |
| Los grupos cambian por completo al repetir el análisis | Grupos **inestables** o inexistentes | Repetir con otras semillas o submuestras; desconfiar si cambian mucho |
| Un grupo de un solo cliente gigante | Valores atípicos (Lección 2.2) | Revisar atípicos, transformar (logaritmo) o usar métodos robustos (8.3, 8.6) |
| Dos clientes "idénticos" que no tienen nada en común | Identificadores o códigos numéricos en la distancia | Quitar IDs, tratar códigos como categorías |
| Con 300 columnas todo parece igual de lejos | Maldición de la dimensionalidad (Lección 2.8) | Reducir dimensiones antes (Lección 8.4) |

### 🔮 Ejercicio 7 — Diagnóstico rápido
Un compañero agrupa clientes por `edad` (años) y `gasto_anual` (pesos, entre 1,000 y 900,000). Te muestra que los 4 grupos son: "gasto muy bajo", "bajo", "medio" y "alto". La edad parece no importar nada. ¿Qué le dices?

✍️ *Mi diagnóstico:*

<details><summary><b>👀 Ver respuesta</b></summary>

Que probablemente **no escaló** los datos: `gasto_anual` tiene números miles de veces más grandes que `edad`, así que la distancia solo "ve" el gasto. Los grupos salen como rebanadas del gasto. Debe escalar y repetir, y además revisar si hay clientes con gastos extremos (un atípico de 900,000 puede quedarse con un grupo para él solo).
</details>

---
## 6. 🧠 Resumen en 5 puntos
1. El aprendizaje **no supervisado** trabaja solo con `X`, sin respuesta: descubre la **estructura** de los datos.
2. Sus cuatro tareas: **agrupar**, **reducir dimensiones**, **detectar anomalías** y **recomendar**.
3. Casi todo se basa en una **distancia**: elegir columnas y **escalarlas** es decidir qué significa "parecido".
4. Un algoritmo de agrupamiento **siempre** devuelve grupos, existan o no.
5. Se evalúa con **métricas internas** (silueta), **validación externa** si hay alguna etiqueta (ARI) y, sobre todo, **utilidad de negocio**.

---
## 7. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 7.1 Repaso rápido (opción múltiple)

**1.** ¿Qué tiene un problema supervisado que **no** tiene uno no supervisado?
- a) Muchas filas · b) Una columna objetivo con la respuesta · c) Columnas numéricas · d) Un modelo

**2.** Agrupas clientes con `ingreso` (pesos) y `num_hijos` sin escalar. ¿Qué esperas?
- a) Que los grupos dependan casi solo del ingreso
- b) Que dependan casi solo del número de hijos
- c) Que ambas columnas pesen igual
- d) Un error de scikit-learn

**3.** Pides a K-means 4 grupos sobre datos sin ninguna estructura. ¿Qué pasa?
- a) Devuelve 1 grupo · b) Lanza una advertencia · c) Devuelve 4 grupos igualmente · d) Devuelve 0 grupos

**4.** Un banco quiere encontrar transacciones sospechosas y no tiene ninguna marcada como fraude. ¿Qué tarea es?
- a) Clasificación · b) Regresión · c) Detección de anomalías · d) Recomendación

**5.** El ARI entre tus grupos y una etiqueta conocida vale 0.02. ¿Qué significa?
- a) Los grupos coinciden casi perfectamente con la etiqueta
- b) Los grupos no tienen casi relación con esa etiqueta
- c) Los grupos son inútiles, seguro
- d) Hay un error en el cálculo

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** La columna objetivo `y` es lo que define al supervisado.
2. **a)** El ingreso tiene números miles de veces más grandes: domina la distancia.
3. **c)** Siempre devuelve los grupos que pidas.
4. **c)** Sin etiquetas de fraude, buscamos lo "raro".
5. **b)** ≈0 significa "como al azar" **respecto a esa etiqueta**. Pero no implica c): los grupos podrían capturar otra cosa útil (por ejemplo, hábitos de compra que no tienen que ver con la etiqueta que elegiste).
</details>

### 7.2 Preguntas para pensar (sin ejecutar)

**A.** Si el aprendizaje no supervisado no tiene "respuesta correcta", ¿cómo sabría una empresa que su segmentación de clientes **funciona**? Propón una forma de comprobarlo en el mundo real.

**B. Caso de análisis 💼** Una cadena de cafeterías agrupa a sus clientes y encuentra 3 grupos: "madrugadores", "de la tarde" y "fin de semana". El director dice: *"El algoritmo lo descubrió, así que es la verdad sobre nuestros clientes."* ¿Estás de acuerdo? ¿Qué le responderías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Con un **experimento**: tratar a cada grupo de forma distinta (una promoción diseñada para cada segmento) y comparar contra un grupo de control que recibe la promoción genérica. Si las ventas o la respuesta mejoran, la segmentación es útil. Otra señal: que los grupos sean **estables** de un mes a otro.

**B.** No del todo. El algoritmo encontró **una** forma de partir los datos según las columnas que le dieron (probablemente la hora y el día de compra). Con otras columnas (gasto, productos) habría encontrado otros grupos. Además, siempre devuelve el número de grupos que le pidas. Los grupos son una **herramienta**, no "la verdad"; su valor se mide por si ayudan a tomar mejores decisiones.
</details>

### 7.3 🧩 Práctica integradora: "Vinos sin etiqueta"

Una bodega tiene 178 vinos analizados en laboratorio (13 medidas químicas: alcohol, acidez, color…). Sabemos que vienen de **3 viñedos**, pero imaginemos que **perdimos** la etiqueta del viñedo. ¿Podemos redescubrir los viñedos solo con la química?

Guardamos la etiqueta real **solo para comprobar al final** (validación externa). El algoritmo nunca la ve.

**Paso 1 ▶️ — Mira los datos**

In [ ]:
vinos = load_wine(as_frame=True)
X_vino = vinos.data                 # 13 medidas químicas
viñedo_real = vinos.target          # 🔒 guardado bajo llave: el algoritmo NO lo usa

print(X_vino.shape)
X_vino.describe().loc[["mean", "min", "max"]].round(1)

✋ Mira la fila `max`. ¿Qué columna tiene los números más grandes? ¿Qué le pasará a la distancia si no escalamos?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**`proline`** (un aminoácido) llega a valores de más de 1,600, mientras que casi todas las demás están por debajo de 30 (`magnesium` llega a ~160). Sin escalar, la distancia la decidirá casi solo `proline`.
</details>

**Paso 2 🔮 — Agrupar sin escalar.** Pedimos 3 grupos a K-means con los datos crudos y comparamos con los viñedos reales con el ARI (1 = perfecto, 0 = al azar). ¿Qué ARI esperas: cerca de 0, de 0.5 o de 1?

✍️ *Mi apuesta:*

In [ ]:
grupos_crudos = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(X_vino)
print(f"ARI sin escalar: {adjusted_rand_score(viñedo_real, grupos_crudos):.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

ARI ≈ **0.37**: algo de relación, pero lejos de redescubrir los viñedos. Los grupos son básicamente "proline bajo / medio / alto".
</details>

**Paso 3 ✏️ — Ahora escalando.** Completa la línea marcada (ya está puesta; compruébalo) y ejecuta.

In [ ]:
X_vino_esc = StandardScaler().fit_transform(X_vino)     # ✏️ el paso que faltaba
grupos_esc = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(X_vino_esc)
print(f"ARI escalando: {adjusted_rand_score(viñedo_real, grupos_esc):.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

ARI ≈ **0.90**: ¡casi redescubrimos los tres viñedos solo con la química! La única diferencia fue **escalar**. Mismo algoritmo, mismos datos, resultado opuesto.
</details>

**Paso 4 ▶️ — ¿Dónde se equivoca?** Una tabla cruzada muestra cuántos vinos de cada viñedo real cayeron en cada grupo.

In [ ]:
pd.crosstab(viñedo_real, grupos_esc, rownames=["Viñedo real"], colnames=["Grupo encontrado"])

✋ Responde:
1. ¿Por qué los números de los grupos (0, 1, 2) no coinciden con los números de los viñedos? ¿Es un error?
2. ¿Cuántos vinos quedaron en un grupo "equivocado"?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **No es un error.** Los números de grupo son nombres arbitrarios: el algoritmo no sabe que existe un "viñedo 0". Lo que importa es que cada viñedo cae casi entero en **un** grupo. (Por eso usamos el ARI y no la exactitud: el ARI no se fija en los nombres.)
2. Solo **6 de 178** vinos: los 3 grupos coinciden casi a la perfección con los viñedos (59 de 59, 65 de 71 y 48 de 48).
</details>

**Paso 5 ✋ — La pregunta de negocio.** Si no hubiéramos tenido la etiqueta, ¿cómo podría la bodega convencerse de que los 3 grupos tienen sentido?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Perfilando cada grupo (promedio de alcohol, color, acidez… por grupo) y preguntando a un enólogo si esos perfiles describen estilos de vino reconocibles. También comprobando que los grupos son **estables** (no cambian al repetir con otra semilla) y que la silueta es razonable. Es exactamente lo que harás en el proyecto del módulo.
</details>

---
## 📚 Para profundizar
- 📘 **Libro:** *Hands-On Machine Learning* (Aurélien Géron), capítulo 9, "Unsupervised Learning Techniques".
- 📺 **Video:** StatQuest, "K-means clustering" (inglés, con subtítulos): un buen adelanto de la próxima lección.
- 📘 **Documentación:** scikit-learn, guía de usuario, sección *Unsupervised learning*.

---
⬅️ **Anterior:** [7.7 Proyecto del módulo: Afinar con presupuesto limitado: las casas de Ames 🎛️](../m07-regularizacion-y-ajuste/7.7-proyecto-del-modulo.md) · ➡️ **Siguiente:** [8.2 K-means: repartir carritos de helado en la playa](8.2-k-means.ipynb)